## Loading Data from Oracle into Spark and Using Splink for Clustering

This notebook demonstrates how to:
1. Connect to an Oracle database using PySpark
2. Load data from Oracle into a Spark DataFrame
3. Use Splink to perform entity resolution/clustering on the data

### Prerequisites

Before running this notebook, ensure you have:

1. **Oracle JDBC Driver**: Download from [Oracle's website](https://www.oracle.com/database/technologies/appdev/jdbc-downloads.html)
   - For Oracle 19c and later: `ojdbc11.jar`
   - For Oracle 12c/18c: `ojdbc8.jar`

2. **Required Python packages**:
```bash
pip install 'splink[spark]'
pip install pyspark
```

3. **Access to an Oracle database** with connection credentials

### Step 1: Configure Spark with Oracle JDBC Driver

In [ ]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from splink.backends.spark import similarity_jar_location

# Path to your Oracle JDBC driver
ORACLE_JDBC_JAR_PATH = "/path/to/ojdbc11.jar"

# Oracle database connection details
ORACLE_HOST = "your-oracle-host"
ORACLE_PORT = "1521"
ORACLE_SERVICE_NAME = "your-service-name"  # or use SID
ORACLE_USER = "your-username"
ORACLE_PASSWORD = "your-password"

# Configure Spark
conf = SparkConf()
conf.set("spark.driver.memory", "12g")
conf.set("spark.default.parallelism", "8")
conf.set("spark.sql.codegen.wholeStage", "false")

# Add Oracle JDBC driver and Splink's similarity JAR
similarity_jar_path = similarity_jar_location()
conf.set("spark.jars", f"{ORACLE_JDBC_JAR_PATH},{similarity_jar_path}")

# Create Spark session
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession(sc)
spark.sparkContext.setCheckpointDir("./tmp_checkpoints")

print("Spark session created successfully!")

### Step 2: Load Data from Oracle Database

There are three main methods to load data from Oracle:

In [ ]:
# Construct Oracle JDBC URL
oracle_jdbc_url = (
    f"jdbc:oracle:thin:@//{ORACLE_HOST}:{ORACLE_PORT}/{ORACLE_SERVICE_NAME}"
)

# Connection properties
connection_properties = {
    "user": ORACLE_USER,
    "password": ORACLE_PASSWORD,
    "driver": "oracle.jdbc.driver.OracleDriver"
}

#### Method 1: Load Entire Table

Simplest method, suitable for small to medium tables:

In [ ]:
# Load entire table
table_name = "YOUR_TABLE_NAME"
df = spark.read.jdbc(
    url=oracle_jdbc_url,
    table=table_name,
    properties=connection_properties
)

df.show(5)

#### Method 2: Load with SQL Query (Recommended)

This allows filtering at the source, reducing data transfer:

In [ ]:
# Load with custom query
query = """
    (SELECT 
        person_id,
        first_name,
        surname,
        TO_CHAR(date_of_birth, 'YYYY-MM-DD') as dob,
        city,
        email
     FROM your_table_name
     WHERE active_flag = 'Y'
       AND date_of_birth IS NOT NULL
    ) AS filtered_data
"""

df = spark.read.jdbc(
    url=oracle_jdbc_url,
    table=query,
    properties=connection_properties
)

print("Data Schema:")
df.printSchema()
print("\nSample Data:")
df.show(5)

#### Method 3: Load with Partitioning (For Very Large Tables)

Enables parallel reads from Oracle, significantly faster for large datasets:

In [ ]:
# Load with partitioning for parallel reads
df = spark.read.jdbc(
    url=oracle_jdbc_url,
    table=table_name,
    column="person_id",  # Numeric column to partition on
    lowerBound=1,          # Minimum value
    upperBound=1000000,    # Maximum value
    numPartitions=10,      # Number of parallel connections
    properties=connection_properties
)

print(f"Number of partitions: {df.rdd.getNumPartitions()}")
df.show(5)

### Step 3: Data Preparation for Splink

In [ ]:
from pyspark.sql import functions as F

# Ensure data types are compatible with Splink
# Convert dates to strings if necessary
df = df.withColumn("dob", F.col("dob").cast("string"))

# Handle NULL values if needed
# df = df.fillna({"email": "", "city": ""})

# Check for required columns
print("Prepared Data Schema:")
df.printSchema()
print("\nData Summary:")
df.describe().show()

### Step 4: Configure Splink for Clustering

Define how Splink should compare records to find duplicates:

In [ ]:
import splink.comparison_library as cl
from splink import Linker, SettingsCreator, SparkAPI, block_on

# Create SparkAPI instance
db_api = SparkAPI(
    spark_session=spark,
    break_lineage_method="checkpoint"  # or "delta_lake_table"
)

# Define comparison settings
settings = SettingsCreator(
    link_type="dedupe_only",  # Deduplication within one dataset
    comparisons=[
        # Compare first names with Jaro-Winkler similarity
        cl.JaroWinklerAtThresholds("first_name", [0.9, 0.7]),
        
        # Compare surnames with Jaro similarity
        cl.JaroAtThresholds("surname", [0.9, 0.7]),
        
        # Compare dates of birth
        cl.DateOfBirthComparison(
            "dob",
            input_is_string=True,
            datetime_metrics=["year", "month"],
            datetime_thresholds=[1, 1],
        ),
        
        # Compare city with exact match and term frequency adjustments
        cl.ExactMatch("city").configure(term_frequency_adjustments=True),
        
        # Compare email addresses
        cl.EmailComparison("email"),
    ],
    blocking_rules_to_generate_predictions=[
        # Only compare records that share the same first name OR surname
        block_on("first_name"),
        block_on("surname"),
    ],
    retain_matching_columns=True,
    retain_intermediate_calculation_columns=True,
)

# Create Linker
linker = Linker(df, settings, db_api)

print("Splink Linker created successfully!")

### Step 5: Train the Splink Model

Splink uses unsupervised learning - no training data required!

In [ ]:
# Estimate probability that two random records match
linker.training.estimate_probability_two_random_records_match(
    [block_on("first_name", "surname")],
    recall=0.7,
)

In [ ]:
# Estimate u probabilities (probability of agreement given non-match)
linker.training.estimate_u_using_random_sampling(max_pairs=1e6)

In [ ]:
# Estimate m probabilities (probability of agreement given match)
# using expectation maximisation
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("first_name", "surname")
)

In [ ]:
# Additional EM training with different blocking rule
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on("dob")
)

### Step 6: Generate Predictions and Create Clusters

In [ ]:
# Predict pairwise links
pairwise_predictions = linker.inference.predict(threshold_match_weight=-10)

# View some predictions
pairwise_predictions_df = pairwise_predictions.as_spark_dataframe()
pairwise_predictions_df.select(
    "match_weight",
    "match_probability",
    "first_name_l",
    "first_name_r",
    "surname_l",
    "surname_r"
).show(10)

In [ ]:
# Cluster predictions to create groups of matching records
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    pairwise_predictions, 
    threshold_match_probability=0.95
)

# Convert to Spark DataFrame
clusters_df = clusters.as_spark_dataframe()

print("Sample Clusters:")
clusters_df.show(20)

### Step 7: Analyze Results

In [ ]:
# Count number of clusters
num_clusters = clusters_df.select("cluster_id").distinct().count()
print(f"Total number of clusters: {num_clusters}")

# Count cluster sizes
print("\nLargest clusters:")
cluster_sizes = clusters_df.groupBy("cluster_id").count()
cluster_sizes.orderBy(F.desc("count")).show(10)

In [ ]:
# Show example of a cluster with multiple records (potential duplicates)
sample_cluster = (
    clusters_df
    .groupBy("cluster_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .select("cluster_id")
    .collect()
)

if sample_cluster:
    cluster_id_value = sample_cluster[0]["cluster_id"]
    print(f"Example cluster with ID: {cluster_id_value}")
    clusters_df.filter(F.col("cluster_id") == cluster_id_value).show(truncate=False)

### Step 8: Save Results

Save the clustered data back to Oracle or to files:

In [ ]:
# Option 1: Save back to Oracle (if you have write permissions)
output_table = "DEDUPLICATED_RECORDS"

clusters_df.write.jdbc(
    url=oracle_jdbc_url,
    table=output_table,
    mode="overwrite",  # or "append"
    properties=connection_properties
)

print(f"Results saved to Oracle table: {output_table}")

In [ ]:
# Option 2: Save to CSV
clusters_df.coalesce(1).write.csv(
    "output/clusters.csv",
    mode="overwrite",
    header=True
)

print("Results saved to CSV")

In [ ]:
# Option 3: Save to Parquet (more efficient for large datasets)
clusters_df.write.parquet(
    "output/clusters.parquet",
    mode="overwrite"
)

print("Results saved to Parquet")

### Optional: Visualize Results with Splink

Splink provides interactive visualizations to understand your linkage:

In [ ]:
# View match weights chart
linker.visualisations.match_weights_chart()

In [ ]:
# View parameter chart
linker.visualisations.comparison_viewer_dashboard(
    pairwise_predictions,
    "output/comparison_viewer.html",
    overwrite=True
)

In [ ]:
# Clean up
spark.stop()
print("Spark session stopped")

## Tips for Production Use

1. **Performance Optimization**:
   - Use partitioning when reading large tables from Oracle
   - Adjust `numPartitions` based on your data size and cluster resources
   - Filter data in Oracle query to reduce data transfer
   - Consider using `break_lineage_method="delta_lake_table"` for better performance with large datasets

2. **Memory Management**:
   - Adjust `spark.driver.memory` and `spark.executor.memory` based on data size
   - Use `.persist()` or `.cache()` strategically on intermediate results
   - Monitor Spark UI for memory usage

3. **Security**:
   - Never hardcode credentials - use environment variables or secrets management
   - Use Oracle Wallet for secure credential storage
   - Enable SSL/TLS for Oracle connections in production

4. **Data Quality**:
   - Clean and standardize data before linking (e.g., trim whitespace, convert to lowercase)
   - Handle NULL values appropriately
   - Validate data types match Splink requirements

5. **Blocking Rules**:
   - Choose blocking rules that significantly reduce comparison space
   - Test different blocking strategies to balance recall and performance
   - Use multiple blocking rules to improve coverage

6. **Model Tuning**:
   - Adjust comparison thresholds based on your data quality
   - Use Splink's interactive tools to diagnose model performance
   - Iterate on blocking rules and comparisons

7. **Monitoring**:
   - Monitor Spark job progress in Spark UI
   - Track cluster statistics (number of clusters, cluster sizes)
   - Set up alerts for job failures or performance degradation